### Προετοιμασία δεδομένων

Το πρώτο κελί προετοιμάζει αυτόματα τα εκπαιδευτικά δεδομένα. Εκτέλεσέ το πριν ξεκινήσεις την άσκηση.


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import json

DATA_ROOT = next((p for p in (Path("../../data"), Path("data"), Path("downloads/python/data")) if p.exists()), Path("data"))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
DATA_URLS = {
    "roads.geojson": "https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp08/roads.geojson",
    "services.csv": "https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp08/services.csv",
    "study_area.geojson": "https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp08/study_area.geojson"
}

def valid_cached(path):
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        if path.suffix.lower() == ".gpkg":
            return path.read_bytes()[:16] == b"SQLite format 3\x00"
        if path.suffix.lower() == ".geojson":
            json.loads(path.read_text(encoding="utf-8"))
        return True
    except Exception:
        return False

for filename, url in DATA_URLS.items():
    target = DATA_ROOT / "gp08" / filename
    if valid_cached(target):
        print(f"[OK] Ήδη διαθέσιμο: {target}")
        continue
    target.parent.mkdir(parents=True, exist_ok=True)
    try:
        with urlopen(url, timeout=30) as response:
            payload = response.read()
        if not payload:
            raise RuntimeError("κενή απάντηση")
        target.write_bytes(payload)
        print(f"[OK] Κατέβηκε: {target}")
    except Exception as exc:
        if target.exists():
            target.unlink()
        print(f"[WARN] Δεν ήταν δυνατή η λήψη {url}: {exc}")


# GP08 — Buffer Analysis

Δημιούργησε buffers γύρω από υπηρεσίες και δρόμους σε projected CRS. Το ερώτημα είναι: τι βρίσκεται μέσα σε συγκεκριμένη απόσταση; Το buffer δημιουργεί νέα Polygon geometry και το distance είναι ακτίνα, όχι διάμετρος.

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
root = DATA_ROOT / "gp08"
services_df = pd.read_csv(root / 'services.csv')
services = gpd.GeoDataFrame(services_df, geometry=gpd.points_from_xy(services_df.longitude, services_df.latitude), crs='EPSG:4326')
roads = gpd.read_file(root / 'roads.geojson')
area = gpd.read_file(root / 'study_area.geojson')


In [ ]:
target_crs = 'EPSG:2100'
services_m = services.to_crs(target_crs)
roads_m = roads.to_crs(target_crs)
area_m = area.to_crs(target_crs)


## Buffer υπηρεσιών
Το 500 είναι μέτρα μόνο όταν το CRS έχει μονάδες μέτρων.

In [ ]:
school_buffers = services_m[services_m['service_type'] == 'school'].copy()
school_buffers['geometry'] = school_buffers.geometry.buffer(500)
school_buffers


In [ ]:
road_buffer = roads_m.geometry.buffer(250)
road_buffer = gpd.GeoDataFrame({'road_id': roads_m['road_id'], 'geometry': road_buffer}, crs=target_crs)
print(school_buffers.geom_type.value_counts())
print(road_buffer.total_bounds)


## QA και επιλογή
Έλεγξε CRS, rows, geometry types και επέλεξε υπηρεσίες μέσα σε buffer σχολείων.

In [ ]:
matched = gpd.sjoin(services_m, school_buffers[['service_id','geometry']], predicate='within', how='inner', lsuffix='service', rsuffix='buffer')
assert services_m.crs.to_epsg() == 2100
assert len(school_buffers) == 2
assert school_buffers.geometry.geom_type.eq('Polygon').all()
assert school_buffers.geometry.notna().all()
assert matched.geometry.geom_type.eq('Point').all()
print('matched services:', len(matched))


In [ ]:
school_buffers.to_file('gp08_school_buffers.gpkg', layer='school_buffers', driver='GPKG')
road_buffer.to_file('gp08_school_buffers.gpkg', layer='road_buffers', driver='GPKG')
print('export complete')
